## <b><u>Infrared.city-1</u>: Tree detection</b>
### <b>Introduction</b>

As urban areas are expanding and global temperatures rise due to climate change, being able to understand how trees can help cool cities has become increasingly important. Furthermore, trees reduce the risk of flooding in cities and promote health and wellbeing. Our project aims at taking the first step in the direction of better tree coverage in urban areas by creating a model which is able to predict tree locations using satellite imagery of densely populated areas. <i>In this notebook we will fetch and prepear the data:</i>

### <b>Notebook structure 2</b>

&nbsp;&nbsp;&nbsp;<b>1. Fetching sentinel and osm data</b><br>
&nbsp;&nbsp;&nbsp;<b>2. Cleaning sentinel and osm data</b><br>
&nbsp;&nbsp;&nbsp;<b>3. Saving cleaned sentinel and osm data</b><br>

### <b>Shared folder structure</b>
/home/jovyan/ideas-dslab-group1-shared/<br>
├── <b>raw data/</b> (backup)<br>
│&ensp;&ensp;&ensp;      ├── baumkataster data/<br>
│&ensp;&ensp;&ensp;     │&ensp;&ensp;&ensp;   ├── vienna_baumkataster<br>
│&ensp;&ensp;&ensp;     │&ensp;&ensp;&ensp;   ├── paris_baumkataster/<br>
│&ensp;&ensp;&ensp;     │&ensp;&ensp;&ensp;   ├── prag_baumkataster/<br>
│&ensp;&ensp;&ensp;     │&ensp;&ensp;&ensp;   └── hamburger_baumkataster/<br>
│&ensp;&ensp;&ensp;   ├── sentinel data/<br>
│&ensp;&ensp;&ensp;   └── osm data/<br>
└── <b>cleaned data/</b><br>
&ensp;&ensp;&ensp; &ensp;├── baumkataster data/<br>
&ensp;&ensp;&ensp;&ensp;          ├── sentinel data/<br>
&ensp;&ensp;&ensp;&ensp;   └── osm data/<br>
<br>

## <b>1. Fetching & preparing the data</b>

### Scraping the data

For our project we have following <a href="https://opentrees.org/#pos=3.7/52.76/0.38">publicly available data</a>:

In another notebook:

1. <b><a href="https://www.data.gv.at/datasets/c91a4635-8b7d-43fe-9b27-d95dec8392a7?locale=en">Vienna Baumkataster</a></b> <i>Official City of Vienna tree cadastre with tree locations (GeoJSON)</i><b>→ continental</b><br>

2. <b><a href="https://opendata.paris.fr/explore/dataset/les-arbres/export/?disjunctive.espece&disjunctive.typeemplacement&disjunctive.arrondissement&disjunctive.genre&disjunctive.libellefrancais&disjunctive.varieteoucultivar&disjunctive.stadedeveloppement&disjunctive.remarquable">Paris Le Arbres</a></b> <i>Official City of Paris tree cadastre with tree locations (GeoJSON)</i><b>→ western / oceanic</b><br>

3. <b><a href="https://opendata-ajuntament.barcelona.cat/data/en/dataset/arbrat-viari">Barcelona</a></b> <i>Official City of Barcelona tree cadastre with tree locations (JSON)</i><b>→ warm / Mediterranean</b><br>

4. <b><a href="https://metaver.de/trefferanzeige?docuuid=C1C61928-C602-4E37-AF31-2D23901E2540">Hamburg Strassenbaumkataster</a></b> <i>Official City of Hambuger tree cadastre with tree locations (GeoJSON)</i><b>→ northern / maritime</b><br>

In this notebook:

5. <b><a href="https://dataspace.copernicus.eu/data-collections/copernicus-sentinel-missions/sentinel-2">Sentinel-2 Satellite Images</a></b> <i>Multispectral satellite imagery including 10 m resolution bands (GeoTIFF)</i><br>


6. <b><a href="https://download.geofabrik.de/europe">OpenStreetMap Urban Features</a></b> <i>OpenStreetMap vector data, including urban context such as buildings and roads ((GeoJSON/shapefile)</i><br>


#  1. Fetching the data
## Sentinel-2 Satellite Images (summer + autumn)

Summer (July) and autumn (October) Sentinel-2 imagery were selected to capture peak vegetation and seasonal decline.

## OpenStreetMap Urban Features

In [ ]:
#2.getting sential-2 satellite images
#B02=blue, B03=green, B04=red, B08=newinfrared
#NDVI = (B08-B04) / (B08+B04)
#summer (July/August) → peak vegetation = full canopy
#autumn (October) → vegetation decline

import numpy as np, rasterio; from rasterio.transform import from_bounds
#from sentinelhub import MosaickingOrder
from sentinelhub import *

cfg=SHConfig()
cfg.sh_client_id="sh-fc6c4793-9f09-43e9-a7b1-047964ba178e" 
cfg.sh_client_secret="7nA9o2no1irzSKX37uIaLiAN8YReBIsF"

cities={"vienna":[16.30,48.17,16.45,48.27],"paris":[2.30,48.84,2.41,48.90],"barcelona":[2.10,41.35,2.22,41.44],"hamburg":[9.90,53.52,10.10,53.60]}
seasons={"summer":("2024-07-01","2024-07-31"),"autumn":("2024-10-01","2024-10-31")}
ev='//VERSION=3 function setup(){return {input:[{bands:["B02","B03","B04","B08","dataMask"]}],output:{bands:5,sampleType:"FLOAT32"}}} function evaluatePixel(s){return [s.B02,s.B03,s.B04,s.B08,s.dataMask]}'

for city,b in cities.items():
    S=[]; box=BBox(bbox=b,crs=CRS.WGS84); size=bbox_to_dimensions(box,10)
    for _,t in seasons.items():
        #added mosaicking_order=MosaickingOrder.LEAST_CC == takes picture with least amount of clouds
        x=SentinelHubRequest(
            evalscript=
            ev,input_data=[SentinelHubRequest.input_data(data_collection=DataCollection.SENTINEL2_L2A,time_interval=t,mosaicking_order=MosaickingOrder.LEAST_CC)],
            responses=[SentinelHubRequest.output_response("default",MimeType.TIFF)],
            bbox=box,size=size,config=cfg).get_data()[0]
        a=x[:,:,:4].astype("float32"); a[x[:,:,4]==0]=np.nan
        ndvi=((a[:,:,3]-a[:,:,2])/(a[:,:,3]+a[:,:,2]+1e-6)).astype("float32")
        S.append(np.dstack([a,ndvi]))
    stack=np.dstack(S); h,w,_=stack.shape; tr=from_bounds(*b,w,h)
    with rasterio.open(f"{city}_summer_autumn_stack.tif","w",driver="GTiff",height=h,width=w,count=stack.shape[2],dtype="float32",crs="EPSG:4326",transform=tr) as dst:
        for i in range(stack.shape[2]):
            dst.write(stack[:,:,i],i+1)